Every DCF starts with a revenue forecast, and most start from the company's own history: recent growth, fading toward the economy. This notebook asks how accurate that is, which simple alternatives do better, and how wide a range around the forecast has to be to be honest.

Each number below is computed in the cell above it from the published results file. The pipeline that builds that file from SEC filings is in [`scripts/run_forecast_accuracy.py`](https://github.com/Courtizy/research/blob/main/scripts/run_forecast_accuracy.py).

In [ ]:
#| label: setup
#| code-summary: "Setup: load the results and the brand chart style"
import json, math, sys
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import Markdown, HTML, display

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "brand"))
import palette                                     # Decision Models colors, validated for color-blind safety

pio.templates["dm"] = palette.plotly_template("valuation", "light")
pio.templates.default = "simple_white+dm"
ORDER = ["blend", "model", "hist3", "sector", "naive"]
COLORS = dict(zip(ORDER, palette.series("valuation", "light", 5)))
NAMES = {"blend": "Blend", "model": "Faded growth", "hist3": "Own 3-yr growth", "sector": "Sector growth",
         "naive": "No change"}

R = json.loads((ROOT / "site" / "data" / "forecast_accuracy.json").read_text())
acc = pd.DataFrame(R["accuracy"])
cov = pd.DataFrame(R["coverage"]).set_index("h")
rng = pd.DataFrame(R["ranges"]).set_index("h")
pct = lambda x: f"{x:.0%}"
tval = lambda d: d["t"] if isinstance(d, dict) else d
cell = lambda r: f'{r.skill_vs_naive:+.0%} (t {abs(tval(r.loss_diff_vs_naive)):.1f})'

## The answer

In [ ]:
#| echo: false
display(Markdown(f"**{R['headline']['lead']}** {R['headline']['sub']}"))
display(HTML('<div class="kpis">' + "".join(f'<div class="kpi"><b>{k["value"]}</b><span>{k["label"]}</span></div>'
                                            for k in R["headline"]["kpis"]) + "</div>"))

## Data and design

- **Universe:** the 300 largest US filers by revenue (at least $1B) in 2011, 2015 and 2019, taken together, so companies that were later acquired or stopped filing stay in. Finance (SIC 60–67) is excluded.
- **Point in time, both ways:** each forecast is made on the day a 10-K was filed, using only facts filed by then, including other companies' filings for the sector median and past misses for the ranges. Each actual is revenue as first reported, so later restatements never leak in.

| Forecast | Revenue *h* years out |
|---|---|
| No change | Base-year revenue. The forecast every method has to beat. |
| Own 3-yr growth | The company's three-year growth rate, held. |
| Sector growth | The median three-year growth of the other companies in its SIC division, each as of its latest 10-K filed by that day. |
| Faded growth | Growth from the company's history (capped at −5% to 20%), fading to 2.5% over ten years. |
| Blend | The geometric mean of no change, faded growth and sector growth. No fitted weights. |

| Range | How it's drawn |
|---|---|
| Simulated | P10–P90 of faded growth with a ±3-point triangular wobble in near-term growth. |
| Calibrated | Quantiles of the blend's past misses, scaled by the company's revenue volatility plus 0.10, at 80%, 90% and 95%. Each forecast from base year 2014 on is scored against quantiles of misses whose actuals had been filed by its own filing date. |

In [ ]:
s = R["sample"]; t = s["range_test"]
pd.Series({"Companies in the universe": s["universe"], "Excluded (finance)": s["excluded_finance"],
           "Companies scored": s["companies_scored"], "Forecast origins": s["origins"],
           "Forecast–outcome pairs": s["observations"],
           "Base years": f'{s["base_years"][0]}–{s["base_years"][1]}',
           "Range test: pairs (base years from " + str(t["first_base_year"]) + ")": t["observations"],
           "Range test: companies": t["companies"]}, name="").to_frame()

## Which forecast is most accurate?

Error is the log ratio of forecast to actual (0.10 is about 10% off). Skill compares mean absolute error with the no-change forecast on the same pairs; the *t* statistic clusters by company.

In [ ]:
#| label: fig-error
#| fig-cap: "Median absolute error by years out."
fig = go.Figure()
for m in ORDER:
    d = acc[acc.method == m]
    fig.add_scatter(x=d.h, y=d.median_abs_error, name=NAMES[m], mode="lines+markers", line_color=COLORS[m])
fig.update_layout(xaxis_title="Years out", yaxis_title="Median absolute log error", height=380,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

In [ ]:
#| code-summary: "Skill against no change, and the blend against faded growth directly"
tab = acc[acc.method != "naive"].assign(cell=lambda d: d.apply(cell, axis=1))
tab = tab.pivot(index="method", columns="h", values="cell").reindex([m for m in ORDER if m != "naive"])
tab = tab.rename(index=NAMES, columns=lambda h: f"{h} yr")
direct = pd.DataFrame(R["blend_vs_model"]).set_index("h")
tab.loc["Blend vs faded growth, directly"] = [f'{g:+.0%} (t {abs(t):.1f})' for g, t in zip(direct.gain, direct.t)]
tab.rename_axis(index=None, columns=None)

### Does it hold across periods?

The sample splits on whether the year being forecast came before 2020 or after.

In [ ]:
#| code-summary: "Skill against no change, by period of the year being forecast"
rows = {}
for k, label in (("targets_before_2020", "before 2020"), ("targets_2020_on", "2020 on")):
    a = pd.DataFrame(R["robustness"][k]["accuracy"])
    for m in ("blend", "model"):
        rows[f"{NAMES[m]}, forecast year {label}"] = {f"{h} yr": cell(r) for h, r in a[a.method == m].set_index("h").iterrows()}
pd.DataFrame(rows).T.rename_axis(columns=None)

## How wide does an honest range have to be?

Both ranges are scored on exactly the same pairs. Dashed lines mark what each promises.

In [ ]:
#| label: fig-coverage
#| fig-cap: "Share of outcomes inside each range, by years out."
S3 = palette.series("valuation", "light", 3)
fig = go.Figure()
fig.add_bar(x=rng.index, y=rng.simulated.map(lambda d: d["share"]), name="Simulated 80% band", marker_color=S3[1])
for lvl, c in (("80", S3[0]), ("90", S3[2])):
    fig.add_bar(x=rng.index, y=rng.calibrated.map(lambda d: d[lvl]["share"]), name=f"Calibrated {lvl}% range", marker_color=c)
for y in (0.8, 0.9):
    fig.add_hline(y=y, line_dash="dash", line_color="#888")
fig.update_layout(barmode="group", xaxis_title="Years out", yaxis_tickformat=".0%", yaxis_range=[0, 1], height=360,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

In [ ]:
#| code-summary: "How far each range reaches below and above the forecast (median)"
rows = []
for h, r in rng.iterrows():
    rows.append({"Years out": h, "Simulated 80% band": f'-{pct(r.simulated["below"])} / +{pct(r.simulated["above"])}',
                 "Calibrated 80%": f'-{pct(r.calibrated["80"]["below"])} / +{pct(r.calibrated["80"]["above"])}',
                 "Calibrated 90%": f'-{pct(r.calibrated["90"]["below"])} / +{pct(r.calibrated["90"]["above"])}',
                 "Held (80 / 90 / 95)": " / ".join(pct(r.calibrated[k]["share"]) for k in ("80", "90", "95"))})
pd.DataFrame(rows).set_index("Years out")

### Does it hold in every slice?

A range can hit 80% on average by being too wide for some companies and too narrow for others. Two checks, three years out:

In [ ]:
#| label: fig-slices
#| fig-cap: "Share inside the calibrated 80% range three years out, by base year (left) and by the company's revenue volatility (right)."
yr, vol = pd.DataFrame(R["ranges_by_year"]), pd.DataFrame(R["ranges_by_volatility"])
fig = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], horizontal_spacing=0.08)
fig.add_bar(x=yr.group, y=yr["80"], marker_color=S3[0], name="Calibrated 80% range", row=1, col=1)
fig.add_bar(x=vol.group, y=vol["80"], marker_color=S3[0], showlegend=False, row=1, col=2)
fig.add_bar(x=yr.group, y=yr.simulated, marker_color=S3[1], name="Simulated 80% band", row=1, col=1)
fig.add_bar(x=vol.group, y=vol.simulated, marker_color=S3[1], showlegend=False, row=1, col=2)
for c in (1, 2):
    fig.add_hline(y=0.8, line_dash="dash", line_color="#888", row=1, col=c)
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.update_layout(barmode="group", height=360, legend=dict(orientation="h", y=-0.2), margin=dict(t=10))
fig.show()

In [ ]:
#| echo: false
low = yr[(yr["80"] < 0.75) & (yr.n >= 30)]
v = vol.set_index("group")["80"]
display(Markdown(
    (f"Base years where the range held under 75%: " + ", ".join(f"{g} ({pct(x)})" for g, x in zip(low.group, low["80"]))
     + ". A range built from the past can't anticipate a kind of shock the record doesn't contain. "
     if len(low) else "Every base year with at least 30 pairs held 75% or more. ")
    + f"By volatility, steady companies' ranges held {pct(v['steady'])}, middle {pct(v['middle'])} and volatile "
    f"{pct(v['volatile'])}."))

## Who gets over-forecast?

In [ ]:
#| label: fig-bias
#| fig-cap: "Mean signed error three years out by prior-growth decile. Above zero = forecast too high; bars are ±2 standard errors."
b = pd.DataFrame(R["bias_by_growth"])
fig = go.Figure()
for m in ["model", "hist3", "blend"]:
    fig.add_scatter(x=b.group, y=b[m].map(lambda x: x["mean"]), name=NAMES[m], mode="lines+markers", line_color=COLORS[m],
                    error_y=dict(array=b[m].map(lambda x: 2 * x["se"]), thickness=1))
fig.add_hline(y=0, line_color="#999")
fig.update_layout(xaxis_title="Prior growth decile (1 = slowest)", yaxis_title="Mean log error", height=360,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

## By sector

In [ ]:
#| code-summary: "Skill against no change three years out, sectors with at least 10 companies"
d = pd.DataFrame(R["by_division"]).set_index("division")
d.assign(**{NAMES[m]: d[m].map(lambda x: f"{x:+.0%}") for m in ("blend", "model", "hist3")})[
    ["companies"] + [NAMES[m] for m in ("blend", "model", "hist3")]]

## Hypotheses, set before looking

In [ ]:
#| echo: false
display(Markdown("\n".join(f"- **{f['label']}.** {f['text']}" for f in R["headline"]["findings"])))

## What it can't tell you

- **The blend's design was chosen with this sample in view.** It has no fitted weights, and the period splits above are a partial check. The ranges are scored out of sample; forecasts made after this study are the real test of the blend.
- **Ranges can't foresee a new kind of shock.** A range built from history is only as wide as history.
- **Large US filers only.** Smaller and private companies are likely more volatile.
- **Revenue only.** Margins, reinvestment and the discount rate add their own uncertainty to a valuation.
- Mechanical forecasts built only from filings. Analysts and management know more.
- Associations, not causes. Analysis of method, not investment advice.

## Reproduce

```bash
pip install -e ".[dev]" && scripts/setup_engine.sh
export SEC_USER_AGENT="Your Name you@example.com"
python scripts/run_forecast_accuracy.py
```